# Рубежный контроль №1
## Группа: ИУ5-66Б
## Студент: [Ваше ФИО]
## Вариант 15
## Задача №2 – Обработка пропусков в данных
## Дополнительное требование: парные диаграммы (pairplot)

## 1. Импорт библиотек и загрузка данных

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

df = pd.read_csv('googleplaystore.csv')
print("Размер датасета:", df.shape)
df.head()

## 2. Первичный анализ и пропуски

In [ ]:
print(df.info())
print("\nПропуски:\n", df.isnull().sum())

## 3. Искусственные пропуски в Category

In [ ]:
np.random.seed(42)
df_work = df.copy()
mask = np.random.random(len(df_work)) < 0.1
df_work.loc[mask, 'Category'] = np.nan
print("Пропуски в Category:", df_work['Category'].isnull().sum())
print("Пропуски в Rating:", df_work['Rating'].isnull().sum())

## 4. Обработка пропусков
Category -> мода; Rating -> медиана

In [ ]:
mode_cat = df_work['Category'].mode()[0]
df_work['Category'].fillna(mode_cat, inplace=True)

median_rating = df_work['Rating'].median()
df_work['Rating'].fillna(median_rating, inplace=True)

print("Пропуски после обработки:", df_work[['Category', 'Rating']].isnull().sum())

## 5. Преобразование числовых признаков

In [ ]:
def convert_size(s):
    if isinstance(s, str):
        s = s.strip()
        if s.endswith('M'):
            return float(s[:-1])
        elif s.endswith('k'):
            return float(s[:-1]) / 1024
    return np.nan

def convert_installs(s):
    if isinstance(s, str):
        s = s.replace(',', '').replace('+', '')
        try:
            return int(s)
        except:
            return np.nan
    return np.nan

def convert_price(s):
    if isinstance(s, str):
        s = s.replace('$', '')
        try:
            return float(s)
        except:
            return np.nan
    return np.nan

df_work['Size_MB'] = df_work['Size'].apply(convert_size)
df_work['Installs_num'] = df_work['Installs'].apply(convert_installs)
df_work['Price_num'] = df_work['Price'].apply(convert_price)

df_num = df_work[['Rating', 'Reviews', 'Size_MB', 'Installs_num', 'Price_num']].dropna()
print("Числовых строк:", df_num.shape)

## 6. Парные диаграммы (pairplot) – требование группы ИУ5-66Б

In [ ]:
sample = df_num.sample(n=5000, random_state=42)
sns.pairplot(sample, diag_kind='hist', plot_kws={'alpha':0.6, 's':20})
plt.suptitle('Парные диаграммы для числовых признаков', y=1.02)
plt.show()

## 7. Выводы о выборе признаков для ML

**Рекомендуемые признаки:**
- Категориальные: Category, Type, Content Rating, Genres
- Количественные: Rating (если не целевая), Reviews, Size_MB, Installs_num, Price_num

**Обоснование:**
- Эти признаки несут информацию о популярности и успешности приложения.
- Категориальные признаки после кодирования (one-hot) позволят учесть нелинейные эффекты.
- Количественные признаки (Reviews и Installs) сильно коррелируют, что требует регуляризации для линейных моделей, но для деревьев не критично.

## 8. Заключение
Работа выполнена: загрузка данных, выявление пропусков, обработка (мода для Category, медиана для Rating), преобразование признаков, построение pairplot, рекомендации по выбору признаков.